In [2]:
#------------------- IMPORT PACKAGES FOR DATA PROCESSING ----------------------#

# Manage datasets
import pandas as pd

# Work with time data
import time 

# Conduct HTTP requests
import requests

# Construct tree structure of HTML data
import html5lib

# Parse HTML data obtained from scraping
from bs4 import BeautifulSoup

# Import webdriver for chrome
from webdriver_manager.chrome import ChromeDriverManager


from selenium import webdriver
from webdriver_manager.chrome import ChromeDriverManager


# Automate navigating within browser (SELENIUM)
#------ Key: Manage keys
#------ Select: Obtain features from website
#------ WebDriverWait: Add wait times implicitly
#------ By: Use common information locator strategies
#------ EC and Options: Browser configuration
#------ remote.command: Check whether browser is active

from selenium import webdriver #to automate the navigating within the browser
from selenium.webdriver.chrome.service import service
from selenium.webdriver.common.keys    import Keys
from selenium.webdriver.support.ui     import Select
from selenium.webdriver.support.ui     import WebDriverWait 
from selenium.webdriver.common.by      import By
from selenium.webdriver.support        import expected_conditions as EC
from selenium.webdriver.chrome.options import Options #to use properties of the chrome webbrowser
from selenium.webdriver.remote.command import Command # Use to check whether the web driver is active

In [3]:

opts = Options()                 # (optional) e.g., opts.add_argument("--headless=new")
driver = webdriver.Chrome(options=opts)


In [4]:
query = "laptop"
url = f"https://www.ebay.com/sch/i.html?_nkw={query}"
driver.get(url)
print(driver.current_url)
print(driver.title)
open("debug_ebay.html","w",encoding="utf-8").write(driver.page_source)


https://www.ebay.com/sch/i.html?_nkw=laptop
Laptop for sale | eBay


2178506

In [5]:

wait = WebDriverWait(driver, 20)

    # (A) Handle cookie/consent if it appears (harmless if it doesn't)
try:
        wait.until(EC.element_to_be_clickable((By.ID, "gdpr-banner-accept"))).click()
        time.sleep(0.5)
except Exception:
    pass
try:
        # some variants use a button with visible text
        btns = driver.find_elements(By.XPATH, '//button[contains(., "Accept")]')
        if btns: btns[0].click()
except Exception:
    pass

In [6]:
wait.until(EC.visibility_of_element_located(
    (By.CSS_SELECTOR, "#srp-river-results li.s-item, #srp-river-results li.s-card")
))
items = driver.find_elements(
    By.CSS_SELECTOR, "#srp-river-results li.s-item, #srp-river-results li.s-card"
)
print("found results:", len(items))


found results: 60


In [11]:
dataset=[]
for i, it in enumerate(items[:60], 1):  # first 60
    title_el = it.find_elements(By.CSS_SELECTOR, ".s-item__title, .s-card__title")
    price_el = it.find_elements(By.CSS_SELECTOR, ".s-item__price, .s-card__price")
    link_el  = it.find_elements(By.CSS_SELECTOR, "a.s-item__link")

    title = title_el[0].text.strip() if title_el else None
    price = price_el[0].text.strip() if price_el else None
    link  = link_el[0].get_attribute("href") if link_el else None

    # skip aggregator tiles
    if not title or title.lower().startswith("shop on ebay"):
        continue
    dataset.append({"title": title, "price": price, "link": link})
    print(f"{i}. {title}\n   {price}\n   {link}\n")
pd.DataFrame(dataset).to_csv("ebay_results.csv", index=False, encoding="utf-8-sig")


1. Lenovo 100e Gen 2 11.6” Laptop Computer Intel 4GB RAM 64GB SSD Windows 11 Pro PC
Opens in a new window or tab
   $131.20
   None

2. HP EliteBook X360 1030 G3 2-in-1 Laptop 13.3” Core i7 16GB 512GB SSD Win 11 Pro
Opens in a new window or tab
   $252.31
   None

3. Lenovo ThinkPad 14” Business Laptop Intel Core i5 16GB RAM 512GB SSD Win 11 Pro
Opens in a new window or tab
   $286.47
   None

4. Lenovo ThinkPad L390 2-in-1 Touch Laptop Core i3 13.3" 8GB RAM 256GB SSD Win 11
Opens in a new window or tab
   $187.53
   None

5. Dell Latitude Laptop PC 11.6" HD Intel Celeron 4GB RAM 64GB SSD Windows 11
Opens in a new window or tab
   $91.17
   None

6. HP EliteBook Laptop Computer PC Intel i5 Up To 32GB RAM 1TB SSD Windows 11
Opens in a new window or tab
   $184.61
   None

7. Dell Latitude Laptop Light Gaming PC Core i7 16GB RAM 512GB SSD Windows 11 Pro
Opens in a new window or tab
   $331.98
   None

8. HP EliteBook 14” FHD AMD Ryzen 5 PRO Laptop PC Up To 32GB RAM 1TB SSD Windows 11
Ope

In [13]:
dataset=[]
for i, it in enumerate(items[:60], 1):  # first 60
    title_el = it.find_elements(By.CSS_SELECTOR, ".s-item__title, .s-card__title")
    price_el = it.find_elements(By.CSS_SELECTOR, ".s-item__price, .s-card__price")


    title = title_el[0].text.strip() if title_el else None
    price = price_el[0].text.strip() if price_el else None
    link = title_el[0].find_element(By.XPATH, "./ancestor::a[1]").get_attribute("href")

    # skip aggregator tiles
    if not title or title.lower().startswith("shop on ebay"):
        continue
    dataset.append({"title": title, "price": price, "link": link})
pd.DataFrame(dataset).to_csv("ebay_results.csv", index=False, encoding="utf-8-sig")